# <b>Trained YOLOv8n Model (with Image)</b>

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

model = YOLO("yolov8n.pt")

result = model.predict("https://ultralytics.com/images/bus.jpg", save=True, conf=0.5)

plots = result[0].plot() 
plots_rgb = cv2.cvtColor(plots, cv2.COLOR_BGR2RGB)  

plt.imshow(plots_rgb)
plt.axis('off')  
plt.show()

# <b>Trained YOLOv8n Model (with Camera)</b>

In [ ]:
from picamera2 import Picamera2, Preview
from ultralytics import YOLO
import cv2
import ipywidgets as widgets
from IPython.display import display
import time

model = YOLO("yolov8n.pt")

picam2 = Picamera2()
camera_config = picam2.create_preview_configuration(
    main={"size": (320, 180), "format": "BGR888"}
)
picam2.configure(camera_config)
picam2.start()

video_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))

display(video_widget)

def convert_to_bytes(image):
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

def update_video_display():
    frame = picam2.capture_array()
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    
    results = model.predict(frame, show=False)
    
    annotated_frame = results[0].plot()  
    
    video_widget.value = convert_to_bytes(annotated_frame)

try:
    while True:
        update_video_display()
        # 이미지 업데이트를 5 FPS로 제한
        time.sleep(0.2)
except KeyboardInterrupt:
    pass

# <b>Camera Capture</b>

In [ ]:
from picamera2 import Picamera2, Preview
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import threading
import time

# 저장할 폴더 경로
save_path = 'dataset/images'
os.makedirs(save_path, exist_ok=True)

class CameraCapture:
    def __init__(self):
        self.picam2 = Picamera2()
        camera_config = self.picam2.create_preview_configuration(
            main={"size": (320, 180), "format": "BGR888"}
        )
        self.picam2.configure(camera_config)
        self.picam2.start()
        
        self.image_count = 0
        self.running = True
        
        # 비디오 스트림을 위한 위젯
        self.video_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))
        display(self.video_widget)
        
        # 캡처 버튼
        self.capture_button = widgets.Button(description="Capture Image")
        self.capture_button.on_click(self.on_capture_button_clicked)
        display(self.capture_button)
        
        # 로그 메시지 위젯
        self.log_output = widgets.Output()
        display(self.log_output)
        
        # 비디오 스트림 업데이트를 위한 스레드
        self.video_thread = threading.Thread(target=self.update_video)
        self.video_thread.start()

    def log(self, message):
        with self.log_output:
            clear_output(wait=True)
            print(message)

    def update_video(self):
        while self.running:
            frame = self.picam2.capture_array()
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

            # JPEG 형식으로 인코딩
            self.video_widget.value = self.convert_to_bytes(frame)

            # 이미지 업데이트를 5 FPS로 제한
            time.sleep(0.2)

    def convert_to_bytes(self, image):
        _, buffer = cv2.imencode('.jpg', image)
        return buffer.tobytes()

    def on_capture_button_clicked(self, b):
        self.log("Capture button clicked")
        frame = self.picam2.capture_array()
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        img_filename = os.path.join(save_path, f'img_{self.image_count}.jpg')
        cv2.imwrite(img_filename, frame)
        self.log(f"Image saved as {img_filename}")
        self.image_count += 1

# 클래스 인스턴스 생성
camera_capture = CameraCapture()


# <b>Run the Trained Sign Detection Model (with bbox)</b>

In [ ]:
from picamera2 import Picamera2, Preview
import cv2
from ultralytics import YOLO
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

model = YOLO("best.pt")

picam2 = Picamera2()
camera_config = picam2.create_preview_configuration(
    main={"size": (320, 180), "format": "BGR888"}
)
picam2.configure(camera_config)
picam2.start()

video_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='320px', height='180px'))
output_widget = widgets.Output()

display(widgets.HBox([video_widget, output_widget]))

def convert_to_bytes(image):
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

def update_video_display():
    frame = picam2.capture_array()
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    
    results = model.predict(frame)
    
    # 예측 추출
    boxes = results[0].boxes.xyxy.numpy()  # 바운딩 박스 좌표
    confidences = results[0].boxes.conf.numpy()  # 신뢰도 점수
    class_ids = results[0].boxes.cls.numpy()  # 클래스 ID

    for i, box in enumerate(boxes):
        conf = confidences[i]
        if conf >= 0.30:  
            x1, y1, x2, y2 = map(int, box)
            cls = int(class_ids[i])
            label = f"Class {cls} ({conf:.2f})"
            color = (0, 255, 0) 
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    video_widget.value = convert_to_bytes(frame)
    
    with output_widget:
        clear_output(wait=True)
        print("Detection Results:")
        for i, conf in enumerate(confidences):
            if conf >= 0.30:
                cls = int(class_ids[i])
                print(f"Class {cls} with confidence {conf:.2f}")

try:
    while True:
        update_video_display()
        # 이미지 업데이트를 5 FPS로 제한
        time.sleep(0.2)
except KeyboardInterrupt:
    pass
